# 💾 Sauvegarde des deux modèles
Exécute ce notebook pour entraîner et sauvegarder `logistic_model.pkl` (Modèle 1) et `logistic_model_reg.pkl` (Modèle 2).

> ⚠️ Place ce fichier dans le **même dossier** que `utils.py`, `ex2data1.txt` et `ex2data2.txt`.

In [1]:
import numpy as np
import math
import joblib
from utils import load_data, map_feature, sig

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def predict(X, w, b):
    m, n = X.shape
    p = np.zeros(m)
    for i in range(m):
        z_wb = np.dot(X[i], w) + b
        p[i] = 1 if sigmoid(z_wb) > 0.5 else 0
    return p

print('✅ Imports OK')

✅ Imports OK


## Modèle 1 — Admission (ex2data1.txt)

In [2]:
def compute_cost(X, y, w, b, lambda_=1):
    m, n = X.shape
    cost = 0
    for i in range(m):
        z = np.dot(X[i], w) + b
        f_wb = sigmoid(z)
        cost += -y[i]*np.log(f_wb) - (1-y[i])*np.log(1-f_wb)
    return cost / m

def compute_gradient(X, y, w, b, lambda_=None):
    m, n = X.shape
    dj_dw = np.zeros(w.shape)
    dj_db = 0.
    for i in range(m):
        f_wb_i = sigmoid(np.dot(X[i], w) + b)
        err_i  = f_wb_i - y[i]
        for j in range(n):
            dj_dw[j] += err_i * X[i, j]
        dj_db += err_i
    return dj_db/m, dj_dw/m

def gradient_descent(X, y, w_in, b_in, cost_fn, grad_fn, alpha, num_iters, lambda_):
    J_history = []
    for i in range(num_iters):
        dj_db, dj_dw = grad_fn(X, y, w_in, b_in, lambda_)
        w_in -= alpha * dj_dw
        b_in -= alpha * dj_db
        if i < 100000:
            J_history.append(cost_fn(X, y, w_in, b_in, lambda_))
        if i % math.ceil(num_iters/10) == 0:
            print(f"Iter {i:5}: cost = {float(J_history[-1]):.4f}")
    return w_in, b_in, J_history

X_train1, y_train1 = load_data('./ex2data1.txt')
m, n = X_train1.shape

np.random.seed(1)
w1 = 0.01 * (np.random.rand(2).reshape(-1,1) - 0.5)
b1 = -8.

w1, b1, _ = gradient_descent(X_train1, y_train1, w1, b1,
                              compute_cost, compute_gradient,
                              alpha=0.001, num_iters=10000, lambda_=0)

acc1 = np.mean(predict(X_train1, w1, b1) == y_train1) * 100
print(f'\n✅ Modèle 1 — Accuracy: {acc1:.1f}%')
joblib.dump({'w': w1, 'b': b1}, 'logistic_model.pkl')
print('💾 logistic_model.pkl sauvegardé')

/tmp/ipykernel_53076/3714651368.py:31: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print(f"Iter {i:5}: cost = {float(J_history[-1]):.4f}")


Iter     0: cost = 0.9638
Iter  1000: cost = 0.3051
Iter  2000: cost = 0.3047
Iter  3000: cost = 0.3044
Iter  4000: cost = 0.3040
Iter  5000: cost = 0.3036
Iter  6000: cost = 0.3033
Iter  7000: cost = 0.3029
Iter  8000: cost = 0.3026
Iter  9000: cost = 0.3022

✅ Modèle 1 — Accuracy: 92.0%
💾 logistic_model.pkl sauvegardé


## Modèle 2 — Microchips régularisé (ex2data2.txt)

In [3]:
def compute_cost_reg(X, y, w, b, lambda_=1):
    m, n = X.shape
    cost = 0
    for i in range(m):
        z = np.dot(X[i], w) + b
        f_wb = sigmoid(z)
        cost += -y[i]*np.log(f_wb) - (1-y[i])*np.log(1-f_wb)
    reg = (lambda_ / (2*m)) * np.sum(w**2)
    return cost/m + reg

def compute_gradient_reg(X, y, w, b, lambda_=1):
    m, n = X.shape
    dj_dw = np.zeros(w.shape)
    dj_db = 0.
    for i in range(m):
        f_wb_i = sigmoid(np.dot(X[i], w) + b)
        err_i  = f_wb_i - y[i]
        for j in range(n):
            dj_dw[j] += err_i * X[i, j]
        dj_db += err_i
    dj_dw = dj_dw/m + (lambda_/m) * w
    return dj_db/m, dj_dw

X_train2, y_train2 = load_data('./ex2data2.txt')
X_mapped = map_feature(X_train2[:, 0], X_train2[:, 1])

np.random.seed(1)
w2 = np.random.rand(X_mapped.shape[1]) - 0.5
b2 = 1.
lambda_ = 0.01

w2, b2, _ = gradient_descent(X_mapped, y_train2, w2, b2,
                              compute_cost_reg, compute_gradient_reg,
                              alpha=0.01, num_iters=10000, lambda_=lambda_)

acc2 = np.mean(predict(X_mapped, w2, b2) == y_train2) * 100
print(f'\n✅ Modèle 2 — Accuracy: {acc2:.1f}%')
joblib.dump({'w': w2, 'b': b2, 'lambda_': lambda_}, 'logistic_model_reg.pkl')
print('💾 logistic_model_reg.pkl sauvegardé')

Iter     0: cost = 0.7210
Iter  1000: cost = 0.5875
Iter  2000: cost = 0.5571
Iter  3000: cost = 0.5332
Iter  4000: cost = 0.5137
Iter  5000: cost = 0.4975
Iter  6000: cost = 0.4838
Iter  7000: cost = 0.4721
Iter  8000: cost = 0.4620
Iter  9000: cost = 0.4531

✅ Modèle 2 — Accuracy: 82.2%
💾 logistic_model_reg.pkl sauvegardé
